# Section B: Alignment, conservation, tree and interpretation

**Pair B:** Gichuru Samuel and Phillip Leo Jnr

How conserved is the Ebola genome across *Ebolavirus* species, and can genome sequence alone separate Bundibugyo from Zaire and Sudan?

This notebook covers stages 4 and 5 of the project.

* **Needs:** `handoff_A.pkl` from Section A (MAFFT and FastTree are installed automatically)
* **Saves:** `handoff_B.pkl`, `alignment.fasta`, `tree.nwk`, the conservation, distance and diagnostic site tables, and the stage 4 and 5 figures

The four section notebooks are run in order: A, then B and C (either order), then D. Run them in the same Colab session, or upload the earlier `handoff_*.pkl` files into `ebola_project/outputs/` first. Keep the settings cell the same in all four notebooks, with your email in `NCBI_EMAIL`.

## Setup

Installs the tools the notebook needs (Biopython, MAFFT, FastTree) and loads the Python libraries.

In [ ]:
import os, sys, re, io, json, time, gzip, shutil, subprocess, pickle, hashlib, zipfile, warnings, random, contextlib, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sys.setrecursionlimit(20000)

IN_COLAB = "google.colab" in sys.modules

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr[-2000:]}")
    return r

try:
    import Bio
except ImportError:
    sh("pip -q install biopython")
    import Bio
if not (shutil.which("mafft") and (shutil.which("fasttree") or shutil.which("FastTree"))):
    print("Installing MAFFT and FastTree (about 1 minute)...")
    sh("apt-get -qq update && apt-get -qq install -y mafft fasttree", check=False)
print("MAFFT:", shutil.which("mafft"), "| FastTree:", shutil.which("fasttree") or shutil.which("FastTree"))
print("CPU cores:", os.cpu_count(), "| Biopython", Bio.__version__)

The settings for the whole run. Put your own email in `NCBI_EMAIL` (NCBI asks for it when downloading). The other settings can stay as they are.

In [ ]:
NCBI_EMAIL = "your_email@example.com"  # put your own email here
NCBI_API_KEY = None
DOWNLOAD_METHOD = "datasets"
ENTREZ_MAX_PER_SPECIES = 2500

MIN_LEN, MAX_LEN = 17000, 20500
MAX_N_FRACTION = 0.01  # drop genomes with more than 1% unknown bases
ZAIRE_CAP_PER_OUTBREAK = 40  # keep at most this many Zaire genomes per outbreak
RANDOM_SEED = 42

EDA_KMER_K = 4
KMER_K = 6  # k-mer length used by the classifiers
ML_CLASSES = ["Bundibugyo", "Sudan", "Zaire"]
FRAGMENT_TEST = True
FRAGMENT_LENGTHS = [250, 500, 1000, 2500]
N_JOBS = -1

ALIGN_MAX_SEQS = 400  # align at most this many genomes
MAFFT_MODE = "fast"  # "fast" or "auto" (slower, more accurate)
TREE_GAMMA = False
REUSE_CACHE = True

WORKDIR = Path("ebola_project")
DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR = [WORKDIR / d for d in ("data", "data/refs", "figures", "outputs")]
for d in (DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

SPECIES = ["Zaire", "Sudan", "Bundibugyo", "Reston", "Tai Forest", "Bombali"]
SPECIES_QUERY = {
    "Zaire": ["Orthoebolavirus zairense", "Zaire ebolavirus"],
    "Sudan": ["Orthoebolavirus sudanense", "Sudan ebolavirus"],
    "Bundibugyo": ["Orthoebolavirus bundibugyoense", "Bundibugyo ebolavirus"],
    "Reston": ["Orthoebolavirus restonense", "Reston ebolavirus"],
    "Tai Forest": ["Orthoebolavirus taiense", "Tai Forest ebolavirus"],
    "Bombali": ["Orthoebolavirus bombaliense", "Bombali ebolavirus"],
}
REF_ACC = {"Zaire": "NC_002549.1", "Sudan": "NC_006432.1", "Bundibugyo": "NC_014373.1",
           "Reston": "NC_004161.1", "Tai Forest": "NC_014372.1", "Bombali": "NC_039345.1"}
GENE_ORDER = ["NP", "VP35", "VP40", "GP", "VP30", "VP24", "L"]
COLORS = {"Zaire": "#1f77b4", "Sudan": "#d62728", "Bundibugyo": "#2ca02c",
          "Reston": "#9467bd", "Tai Forest": "#ff7f0e", "Bombali": "#8c564b"}

TIMINGS, RESULTS = [], {}
@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"> {label} ...")
    try: yield
    finally:
        dt = time.time() - t; TIMINGS.append((label, dt)); print(f"< {label}: {dt/60:.1f} min ({dt:.0f} s)")

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

if "example.com" in NCBI_EMAIL:
    raise ValueError("Set NCBI_EMAIL (above) to your real email address before running the real download.")

Helper functions for reading and writing FASTA files, finding the genes in a reference genome and downloading data from NCBI.

In [ ]:
def read_fasta(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as fh:
        name, desc, buf = None, "", []
        for line in fh:
            if line.startswith(">"):
                if name is not None: yield name, desc, "".join(buf).upper()
                head = line[1:].strip().split(None, 1)
                name, desc, buf = head[0], (head[1] if len(head) > 1 else ""), []
            else: buf.append(line.strip())
        if name is not None: yield name, desc, "".join(buf).upper()

def write_fasta(path, items, width=80):
    with open(path, "w") as fh:
        for name, seq in items:
            fh.write(f">{name}\n")
            for i in range(0, len(seq), width): fh.write(seq[i:i+width] + "\n")

def base_acc(a): return a.split(".")[0]

def canon_gene(q):
    g = (q.get("gene", [""])[0]).strip(); prod = (q.get("product", [""])[0]).lower()
    if g.upper() in GENE_ORDER: return g.upper()
    if "nucleoprotein" in prod: return "NP"
    if "vp35" in prod or "cofactor" in prod: return "VP35"
    if "vp40" in prod or "matrix" in prod: return "VP40"
    if "vp30" in prod: return "VP30"
    if "vp24" in prod: return "VP24"
    if "glycoprotein" in prod and "secreted" not in prod and "small" not in prod: return "GP"
    if "polymerase" in prod or prod.strip() == "l": return "L"
    return None

def gene_table(rec):
    spans = {}
    for f in rec.features:
        if f.type != "CDS": continue
        g = canon_gene(f.qualifiers)
        if g is None: continue
        s, e = int(f.location.start) + 1, int(f.location.end)
        a, b = spans.get(g, (s, e)); spans[g] = (min(a, s), max(b, e))
    df = pd.DataFrame([(g, *spans[g]) for g in GENE_ORDER if g in spans], columns=["gene", "start", "end"])
    missing = [g for g in GENE_ORDER if g not in spans]
    if missing: print(f"  WARNING {rec.id}: genes not found in annotation: {missing}")
    return df

def fetch_refs():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    refs, genes = {}, {}
    for sp, acc in REF_ACC.items():
        p = REF_DIR / f"{base_acc(acc)}.gb"
        if not p.exists():
            with Entrez.efetch(db="nuccore", id=base_acc(acc), rettype="gb", retmode="text") as h: p.write_text(h.read())
            time.sleep(0.4)
        rec = SeqIO.read(p, "genbank")
        refs[sp] = str(rec.seq).upper(); genes[sp] = gene_table(rec)
        REF_ACC[sp] = rec.id
    return refs, genes

def jget(d, *keys):
    for k in keys:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def download_datasets_cli():
    ds = DATA_DIR / "datasets"
    if not ds.exists():
        sh(f"curl -sSL -o {ds} https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets && chmod +x {ds}")
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        got = False
        for nm in names:
            z = DATA_DIR / f"{sp.replace(' ', '_')}.zip"
            r = sh(f'{ds} download virus genome taxon "{nm}" --complete-only --filename {z}', check=False)
            if r.returncode != 0 or not z.exists(): continue
            out = DATA_DIR / f"unz_{sp.replace(' ', '_')}"; shutil.rmtree(out, ignore_errors=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
            fna = next(out.rglob("genomic.fna"), None); rep = next(out.rglob("data_report.jsonl"), None)
            if fna is None: continue
            seqs = {n: (d, s) for n, d, s in read_fasta(fna)}
            reports = {}
            if rep is not None:
                for line in open(rep):
                    j = json.loads(line); reports[j.get("accession")] = j
            for acc, (desc, s) in seqs.items():
                j = reports.get(acc, {})
                rows.append(dict(accession=acc, species=sp, isolate=jget(j, "isolate", "name") or desc,
                                 collection_date=jget(j, "isolate", "collectionDate"),
                                 geo=jget(j, "location", "geographicLocation"), region=jget(j, "location", "geographicRegion"),
                                 desc=desc))
                records[acc] = s
            print(f"  {sp}: {len(seqs)} genomes (query '{nm}')"); got = True; break
        if not got: print(f"  WARNING: nothing downloaded for {sp}")
    return records, pd.DataFrame(rows)

def download_entrez():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        for nm in names:
            term = f'"{nm}"[Organism] AND {MIN_LEN}:{MAX_LEN}[SLEN]'
            res = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=0, usehistory="y"))
            n = int(res["Count"])
            if n > 0: break
        n = min(n, ENTREZ_MAX_PER_SPECIES); print(f"  {sp}: fetching {n} records")
        for start in range(0, n, 200):
            for attempt in range(4):
                try:
                    h = Entrez.efetch(db="nuccore", rettype="gb", retmode="text", retstart=start, retmax=min(200, n - start),
                                      webenv=res["WebEnv"], query_key=res["QueryKey"])
                    batch = list(SeqIO.parse(h, "genbank")); break
                except Exception as ex:
                    print("   retry", attempt, ex); time.sleep(3); batch = []
            for rec in batch:
                src = next((f for f in rec.features if f.type == "source"), None); q = src.qualifiers if src else {}
                records[rec.id] = str(rec.seq).upper()
                rows.append(dict(accession=rec.id, species=sp, isolate=(q.get("isolate") or q.get("strain") or [""])[0],
                                 collection_date=(q.get("collection_date") or [None])[0], geo=(q.get("country") or [None])[0],
                                 region=None, desc=rec.description))
            time.sleep(0.4)
    return records, pd.DataFrame(rows)

Functions for counting k-mers, used again in later sections.

In [ ]:
_LUT = np.full(256, -1, dtype=np.int64)
for _i, _c in enumerate("ACGT"): _LUT[ord(_c)] = _i

def kmer_index(seq, k):
    a = _LUT[np.frombuffer(seq.encode(), dtype=np.uint8)]
    n = len(a) - k + 1
    if n <= 0: return np.zeros(0, dtype=np.int64)
    idx = np.zeros(n, dtype=np.int64); bad = np.zeros(n, dtype=bool)
    for j in range(k):
        s = a[j:j+n]; bad |= s < 0; idx = idx * 4 + s
    idx[bad] = -1
    return idx

def kmer_freq(seq, k):
    idx = kmer_index(seq, k); idx = idx[idx >= 0]
    c = np.bincount(idx, minlength=4**k).astype(np.float32)
    return c / max(c.sum(), 1)

def kmer_matrix(seqs, k):
    return np.vstack([kmer_freq(s, k) for s in seqs])

def kmer_labels(k): return ["".join(p) for p in itertools.product("ACGT", repeat=k)]

from sklearn.preprocessing import StandardScaler

Loads the cleaned genomes and reference genomes saved by Section A.

In [ ]:
_hA = pickle.load(open(OUT_DIR / "handoff_A.pkl", "rb"))
qc, SEQ, REFS, GENES = _hA["qc"], _hA["SEQ"], _hA["REFS"], _hA["GENES"]
REF_ACC.update(_hA["REF_ACC"])
sp_of = dict(zip(qc.accession, qc.species)); ob_of = dict(zip(qc.accession, qc.outbreak))
print(f"Loaded Section A results: {len(qc)} genomes after quality control")
display(qc.species.value_counts().rename("genomes").to_frame().T)

## 4. Alignment, conservation and phylogenetic tree

Lines up the genomes so that matching positions can be compared, measures how conserved each position is, and builds a tree. This is the slowest part of the notebook, so the results are saved for re-runs.

Chooses which genomes to align, keeping every outbreak represented, and aligns them with MAFFT.

In [ ]:
def select_for_alignment(qc, max_seqs, seed):
    if max_seqs is None or len(qc) <= max_seqs: return list(qc.accession)
    rng = np.random.default_rng(seed)
    keep = list(qc.loc[qc.is_ref, "accession"]); rest = qc[~qc.is_ref]
    pools = {k: list(rng.permutation(v.accession.values)) for k, v in rest.groupby("outbreak")}
    while len(keep) < max_seqs and any(pools.values()):
        for k in list(pools):
            if pools[k] and len(keep) < max_seqs: keep.append(pools[k].pop())
    return keep

aln_ids_in = select_for_alignment(qc, ALIGN_MAX_SEQS, RANDOM_SEED)
print(f"Aligning {len(aln_ids_in)} of {len(qc)} genomes")
write_fasta(OUT_DIR / "to_align.fasta", [(a, SEQ[a]) for a in aln_ids_in])
ALN_FA = OUT_DIR / "alignment.fasta"

with timed("Stage 4a: MAFFT alignment"):
    if REUSE_CACHE and ALN_FA.exists() and ALN_FA.stat().st_size > 0:
        print("Using cached alignment", ALN_FA)
    else:
        opts = "--auto" if MAFFT_MODE == "auto" else "--retree 2 --maxiterate 0"
        sh(f"mafft --thread {os.cpu_count() or 2} {opts} --quiet {OUT_DIR/'to_align.fasta'} > {ALN_FA}")

aln_ids, aln_rows = [], []
for n, d, s in read_fasta(ALN_FA): aln_ids.append(n); aln_rows.append(np.frombuffer(s.encode(), dtype=np.uint8))
ALN = np.vstack(aln_rows); n_seq, n_col = ALN.shape
print(f"Alignment: {n_seq} sequences x {n_col} columns")
CODES = _LUT[ALN].astype(np.int8)
sp_of = dict(zip(qc.accession, qc.species)); ob_of = dict(zip(qc.accession, qc.outbreak))
aln_sp = np.array([sp_of[a] for a in aln_ids])

Measures how conserved each position is, within each species and across species. The results are shown on the Zaire reference genome and summarised for each gene.

In [ ]:
with timed("Stage 4b: conservation"):
    def col_counts(codes):
        return np.stack([(codes == b).sum(0) for b in range(4)]).astype(np.float64)
    def entropy_from_counts(cnt):
        nv = cnt.sum(0); p = cnt / np.maximum(nv, 1)
        with np.errstate(divide="ignore", invalid="ignore"):
            h = -np.where(p > 0, p * np.log2(p), 0).sum(0)
        return np.where(nv > 0, h, np.nan), nv

    cnt_all = col_counts(CODES); ent_all, nv_all = entropy_from_counts(cnt_all)
    cons_all = 1 - ent_all / 2
    gap_frac = 1 - nv_all / n_seq

    sp_present = [s for s in SPECIES if (aln_sp == s).any()]
    sp_cons, within = [], {}
    for s in sp_present:
        rows = CODES[aln_sp == s]; c = col_counts(rows); nv = c.sum(0); ok = nv >= 0.5 * len(rows)
        sp_cons.append(np.where(ok, c.argmax(0), -1))
        e, _ = entropy_from_counts(c); within[s] = np.where(ok, 1 - e / 2, np.nan)
    sp_cons = np.vstack(sp_cons); n_sp = (sp_cons >= 0).sum(0)
    cnt_sp = np.stack([(sp_cons == b).sum(0) for b in range(4)]).astype(np.float64)
    ent_sp, _ = entropy_from_counts(cnt_sp)
    cons_sp = np.where(n_sp >= 3, 1 - ent_sp / 2, np.nan)
    invariant_sp = (n_sp >= min(4, len(sp_present))) & (cnt_sp.max(0) == n_sp)
    fully_cons_all = (nv_all >= 0.9 * n_seq) & (ent_all == 0)

    zid = next(a for a in aln_ids if base_acc(a) == base_acc(REF_ACC["Zaire"])); zi = aln_ids.index(zid)
    ref_cols = np.flatnonzero(ALN[zi] != ord("-"))
    print(f"Zaire reference: {len(ref_cols)} bases in alignment vs {len(REFS['Zaire'])} in the sequence")
    col2ref = np.full(n_col, -1); col2ref[ref_cols] = np.arange(len(ref_cols))
    R = lambda x: np.asarray(x)[ref_cols]
    cons_sp_r, cons_all_r, inv_sp_r, full_r = R(cons_sp), R(cons_all), R(invariant_sp), R(fully_cons_all)

    gt = GENES["Zaire"]
    rows = []
    cod = np.zeros(len(ref_cols), bool)
    for g in gt.itertuples():
        sl = slice(g.start - 1, g.end); cod[sl] = True
        rows.append(dict(gene=g.gene, length=g.end - g.start + 1, cons_across_species=np.nanmean(cons_sp_r[sl]),
                         pct_invariant_across_species=100 * inv_sp_r[sl].mean(), cons_all_genomes=np.nanmean(cons_all_r[sl]),
                         pct_identical_in_all_genomes=100 * full_r[sl].mean(),
                         **{f"within_{s}": np.nanmean(R(within[s])[sl]) for s in sp_present}))
    rows.append(dict(gene="non-coding", length=int((~cod).sum()), cons_across_species=np.nanmean(cons_sp_r[~cod]),
                     pct_invariant_across_species=100 * inv_sp_r[~cod].mean(), cons_all_genomes=np.nanmean(cons_all_r[~cod]),
                     pct_identical_in_all_genomes=100 * full_r[~cod].mean(), **{f"within_{s}": np.nanmean(R(within[s])[~cod]) for s in sp_present}))
    gene_cons = pd.DataFrame(rows).round(3); gene_cons.to_csv(OUT_DIR / "gene_conservation.csv", index=False)
    RESULTS["genome_invariant_across_species_pct"] = float(100 * inv_sp_r.mean())

display(gene_cons)
win = lambda x, w=150: pd.Series(x).rolling(w, center=True, min_periods=1).mean().values
fig, ax = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
ax[0].plot(win(cons_sp_r), color="black", lw=1.2, label="across species (consensus of each species)")
ax[0].plot(win(cons_all_r), color="grey", lw=.9, alpha=.8, label="all genomes pooled (Zaire-weighted)")
for s in ML_CLASSES:
    if s in within: ax[1].plot(win(R(within[s])), color=COLORS[s], lw=1.1, label=f"within {s}")
for a in ax:
    for i, g in enumerate(gt.itertuples()):
        a.axvspan(g.start, g.end, color="#dddddd" if i % 2 == 0 else "#bbbbbb", alpha=.35, lw=0)
for g in gt.itertuples(): ax[0].text((g.start + g.end) / 2, 1.0, g.gene, ha="center", va="bottom", fontsize=9)
ax[0].set_ylabel("conservation (150-nt window)"); ax[1].set_ylabel("within-species conservation"); ax[1].set_xlabel("position on the Zaire reference genome (nt)")
ax[0].legend(loc="lower left", fontsize=8); ax[1].legend(loc="lower left", fontsize=8); ax[0].set_title("Conservation along the genome (1 = invariant)", pad=18)
savefig("04_conservation_profile")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
g7 = gene_cons[gene_cons.gene != "non-coding"]
ax[0].bar(g7.gene, g7.cons_across_species, color="#4c72b0"); ax[0].set_ylim(g7.cons_across_species.min() * .9, 1); ax[0].set_title("Mean across-species conservation per gene")
ax[1].bar(g7.gene, g7.pct_invariant_across_species, color="#55a868"); ax[1].set_title("% of sites identical in every species")
savefig("04_gene_conservation")

Builds a phylogenetic tree from the alignment with FastTree.

In [ ]:
with timed("Stage 4c: phylogenetic tree (FastTree)"):
    gap_cut = 0.9
    keep_cols = gap_frac <= gap_cut
    TRIM_FA, TREE_NWK = OUT_DIR / "alignment_trimmed.fasta", OUT_DIR / "tree.nwk"
    if REUSE_CACHE and TREE_NWK.exists() and TREE_NWK.stat().st_size > 0:
        print("Using cached tree", TREE_NWK)
    else:
        write_fasta(TRIM_FA, [(a, ALN[i, keep_cols].tobytes().decode()) for i, a in enumerate(aln_ids)])
        ft = shutil.which("fasttree") or shutil.which("FastTree")
        sh(f"{ft} -nt -gtr {'-gamma' if TREE_GAMMA else ''} -quiet {TRIM_FA} > {TREE_NWK}")
    print(f"Tree built on {int(keep_cols.sum())} of {n_col} alignment columns")

Draws the tree, checks whether each species forms its own branch, and finds the species closest to Bundibugyo.

In [ ]:
from Bio import Phylo
from matplotlib.collections import LineCollection

tree = Phylo.read(str(TREE_NWK), "newick"); tree.root_at_midpoint(); tree.ladderize()
tips = tree.get_terminals()
tips_by_sp = {s: [t for t in tips if sp_of[t.name] == s] for s in sp_present}

mono_rows = []
for s, ts in tips_by_sp.items():
    m = tree.is_monophyletic(ts); is_mono = m is not False
    mrca = ts[0] if len(ts) == 1 else tree.common_ancestor(ts)
    n_under = len(mrca.get_terminals()) if not mrca.is_terminal() else 1
    mono_rows.append(dict(species=s, tips=len(ts), monophyletic=is_mono, tips_under_MRCA=n_under,
                          support=(None if mrca.is_terminal() or mrca.confidence is None else round(float(mrca.confidence), 2))))
mono = pd.DataFrame(mono_rows); display(mono)

def sister_of(species):
    ts = tips_by_sp[species]; mrca = ts[0] if len(ts) == 1 else tree.common_ancestor(ts)
    path = tree.get_path(mrca); parent = path[-2] if len(path) >= 2 else tree.root
    sibs = [c for c in parent.clades if c is not mrca]
    comp = pd.Series([sp_of[t.name] for c in sibs for t in c.get_terminals()]).value_counts()
    return comp
BUND_SISTER = sister_of("Bundibugyo") if "Bundibugyo" in tips_by_sp else None
print("Sister group of Bundibugyo (species composition):"); print(BUND_SISTER.to_string() if BUND_SISTER is not None else "n/a")

def plot_tree(tree, fname):
    depth = tree.depths(); ypos = {}
    for i, t in enumerate(tree.get_terminals()): ypos[t] = i
    def assign(c):
        if c.is_terminal(): return ypos[c]
        ys = [assign(ch) for ch in c.clades]; ypos[c] = (min(ys) + max(ys)) / 2; return ypos[c]
    assign(tree.root)
    spset = {}
    def sets(c):
        spset[c] = {sp_of[c.name]} if c.is_terminal() else set().union(*[sets(ch) for ch in c.clades]); return spset[c]
    sets(tree.root)
    segs, cols = [], []
    def col(c): return COLORS[next(iter(spset[c]))] if len(spset[c]) == 1 else "#888888"
    def walk(c):
        for ch in c.clades:
            segs.append([(depth[c], ypos[ch]), (depth[ch], ypos[ch])]); cols.append(col(ch)); walk(ch)
        if c.clades:
            ys = [ypos[ch] for ch in c.clades]; segs.append([(depth[c], min(ys)), (depth[c], max(ys))]); cols.append(col(c))
    walk(tree.root)
    fig, ax = plt.subplots(figsize=(11, 12))
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=1.1))
    ax.set_xlim(0, max(depth.values()) * 1.03); ax.set_ylim(-1, len(tips) + 1); ax.set_yticks([])
    ax.set_xlabel("substitutions per site"); ax.set_title("Maximum-likelihood tree (FastTree GTR, midpoint-rooted); branch colour = species (grey = mixed)")
    for s in sp_present: ax.plot([], [], color=COLORS[s], lw=3, label=f"{s} (n={len(tips_by_sp[s])})")
    ax.legend(loc="lower left", fontsize=9); savefig(fname)
with timed("Stage 4c: tree plot"):
    plot_tree(tree, "04_phylogenetic_tree")
RESULTS["tree_monophyly"] = {r.species: bool(r.monophyletic) for r in mono.itertuples()}
RESULTS["bundibugyo_sister"] = None if BUND_SISTER is None else BUND_SISTER.to_dict()

## 5. Interpretation

What the alignment tells us about how the species differ.

Measures how different the species are from each other. Then it looks for diagnostic sites: positions where Bundibugyo has a different base from both Zaire and Sudan, while each of the three species is consistent within itself.

In [ ]:
with timed("Stage 5: distances and diagnostic sites"):
    rng = np.random.default_rng(RANDOM_SEED)
    idx_by_sp = {s: np.flatnonzero(aln_sp == s) for s in sp_present}
    samp = {s: rng.choice(v, size=min(len(v), 25), replace=False) for s, v in idx_by_sp.items()}
    sel = np.concatenate([samp[s] for s in sp_present]); lab = np.concatenate([[s] * len(samp[s]) for s in sp_present])
    sub = CODES[sel]; D = np.zeros((len(sel), len(sel)))
    for i in range(len(sel)):
        v = (sub >= 0) & (sub[i] >= 0); D[i] = ((sub != sub[i]) & v).sum(1) / np.maximum(v.sum(1), 1)
    pd_mat = pd.DataFrame([[D[np.ix_(lab == a, lab == b)][~np.eye((lab == a).sum(), (lab == b).sum(), dtype=bool)].mean()
                            if (a == b and (lab == a).sum() > 1) or a != b else np.nan for b in sp_present] for a in sp_present],
                          index=sp_present, columns=sp_present)
    pd_mat.to_csv(OUT_DIR / "species_pdistance.csv")
    plt.figure(figsize=(7, 5.5)); sns.heatmap(pd_mat * 100, annot=True, fmt=".1f", cmap="viridis_r", cbar_kws={"label": "% nucleotide difference"})
    plt.title("Mean pairwise difference between species (p-distance)"); savefig("05_species_pdistance")

    cls_base, cls_major = {}, {}
    for c in ML_CLASSES:
        rows_c = CODES[aln_sp == c]; cc = col_counts(rows_c); nv = cc.sum(0); ok = nv >= 0.8 * len(rows_c)
        cls_base[c] = np.where(ok, cc.argmax(0), -1); cls_major[c] = np.where(ok, cc.max(0) / np.maximum(nv, 1), 0)
    allok = np.all([cls_base[c] >= 0 for c in ML_CLASSES], axis=0)
    def diag_sites(minfix):
        fixed = np.all([cls_major[c] >= minfix for c in ML_CLASSES], axis=0) & allok
        return {"Bundibugyo vs Zaire+Sudan": fixed & (cls_base["Bundibugyo"] != cls_base["Zaire"]) & (cls_base["Bundibugyo"] != cls_base["Sudan"]),
                "Zaire vs Sudan": fixed & (cls_base["Zaire"] != cls_base["Sudan"])}
    D100, D95 = diag_sites(0.999), diag_sites(0.95)
    drows = []
    for name in D100:
        for tag, dd in (("fixed (100%)", D100), (">=95% within class", D95)):
            cols = np.flatnonzero(dd[name]); rp = col2ref[cols]; rp = rp[rp >= 0]
            per_gene = {g.gene: int(((rp >= g.start - 1) & (rp < g.end)).sum()) for g in gt.itertuples()}
            drows.append(dict(comparison=name, criterion=tag, total_sites=int(dd[name].sum()), **per_gene))
    diag = pd.DataFrame(drows); display(diag); diag.to_csv(OUT_DIR / "diagnostic_sites.csv", index=False)
    RESULTS["diag_bund_fixed"] = int(D100["Bundibugyo vs Zaire+Sudan"].sum()); RESULTS["diag_bund_95"] = int(D95["Bundibugyo vs Zaire+Sudan"].sum())
    RESULTS["diag_zs_fixed"] = int(D100["Zaire vs Sudan"].sum())

ranked = gene_cons[gene_cons.gene != "non-coding"].sort_values("cons_across_species", ascending=False)
print("\nDraft reading of the core analysis (edit before submitting):")
print(f"* Most conserved gene across species: {ranked.iloc[0].gene} ({ranked.iloc[0].cons_across_species:.3f}); least conserved: {ranked.iloc[-1].gene} ({ranked.iloc[-1].cons_across_species:.3f}).")
nc = gene_cons[gene_cons.gene == 'non-coding'].iloc[0]
print(f"* Non-coding regions: mean across-species conservation {nc.cons_across_species:.3f} versus {np.nanmean(cons_sp_r[cod]):.3f} in coding regions.")
print(f"* {RESULTS['genome_invariant_across_species_pct']:.1f}% of reference positions are identical in every species' consensus.")
for r in mono.itertuples():
    print(f"* {r.species}: {'forms a single clade' if r.monophyletic else 'is NOT monophyletic (' + str(r.tips_under_MRCA) + ' tips under its MRCA vs ' + str(r.tips) + ')'}" + ("" if r.support is None else f", support {r.support}"))
if BUND_SISTER is not None: print(f"* Bundibugyo's sister group in the tree: {', '.join(f'{k} ({v})' for k, v in BUND_SISTER.items())}")

Saves the results so that Section D can use them.

In [ ]:
pickle.dump(dict(gene_cons=gene_cons, cons_sp_r=cons_sp_r, diag=diag, mono=mono, BUND_SISTER=BUND_SISTER, ranked=ranked,
                 cod=cod, sp_present=sp_present, n_seq=n_seq, RESULTS=RESULTS, TIMINGS=TIMINGS),
            open(OUT_DIR / "handoff_B.pkl", "wb"))
print("Saved", OUT_DIR / "handoff_B.pkl")